# SAP Medallion — Lakeflow Declarative Pipeline
## Pipeline Bronze → Silver → Gold sobre datos NYC Taxi (analogía SAP)

**IMPORTANTE:** Este notebook se usa como source code de un Lakeflow Pipeline.
NO lo ejecutes como notebook normal — impórtalo en Workflows → Pipelines.

---

### Analogía con el curso SAP

| Este pipeline | Equivalente SAP del curso |
|---------------|---------------------------|
| `samples.nyctaxi.trips` | `companycode_share` via BDC Delta Sharing |
| `bronze_trips` | `vbak_bronze` |
| `silver_trips` | `vbak_silver` con validaciones |
| `gold_kpis_by_zone` | `gold_sales_kpis` por org de ventas |
| pickup_zip | VKORG (organización de ventas) |
| fare_amount | NETWR (valor neto de la orden) |


## Bronze — Streaming Table

```
Equivalente: Auto Loader desde BDC Delta Sharing
En producción SAP: STREAM(sap_bdc_catalog.sales.SalesOrder)
```

In [0]:
# Bronze: ingesta de datos crudos
# En producción SAP: reemplazar samples.nyctaxi.trips
# por el share de BDC: STREAM(sap_bdc_catalog.sales.SalesOrder)

import dlt
from pyspark.sql.functions import current_timestamp, lit, col, year, month, round as _round

@dlt.table(
    name='bronze_trips',
    comment='Datos crudos de viajes NYC — equivalente a vbak_bronze en SAP'
)
def bronze_trips():
    return (
        spark.readStream
            .format('delta')
            .table('samples.nyctaxi.trips')
            .withColumn('_ingested_at', current_timestamp())
            .withColumn('_source', lit('nyctaxi'))
    )


## Silver — Materialized View con validaciones de calidad

```
Equivalente: vbak_silver con MERGE + validaciones
DLT agrega expect() para calidad de datos automática
```

In [0]:
@dlt.table(
    name='silver_trips',
    comment='Viajes validados y enriquecidos — equivalente a vbak_silver SAP'
)
@dlt.expect('fare_positivo', 'fare_amount > 0')
@dlt.expect('distancia_positiva', 'trip_distance > 0')
@dlt.expect_or_drop('pickup_zip_valido', 'pickup_zip IS NOT NULL')
def silver_trips():
    return (
        dlt.read_stream('bronze_trips')
            .select(
                col('pickup_zip'),
                col('dropoff_zip'),
                col('fare_amount'),
                col('trip_distance'),
                _round(col('fare_amount') / col('trip_distance'), 2).alias('fare_per_mile'),
                col('tpep_pickup_datetime').cast('date').alias('pickup_date'),
                year(col('tpep_pickup_datetime')).alias('year'),
                month(col('tpep_pickup_datetime')).alias('month'),
                col('_ingested_at')
            )
    )


### Las expectativas de calidad — equivalente a validaciones SAP

```
@dlt.expect         → registra el error pero deja pasar la fila
@dlt.expect_or_drop → descarta la fila si no cumple
@dlt.expect_or_fail → detiene el pipeline si no cumple

Equivalente en el curso:
  WHERE AUART = 'TA'           → solo órdenes estándar
  WHERE NETWR > 0              → sin valores nulos
  WHERE ERDAT IS NOT NULL      → fecha obligatoria

En BDC: los Data Products ya vienen con calidad garantizada por SAP
En Databricks: defines las reglas tú
```


## Gold — Materialized View con KPIs de negocio

```
Equivalente: gold_sales_kpis por org de ventas
```

In [0]:
from pyspark.sql.functions import count, sum as _sum, avg

@dlt.table(
    name='gold_kpis_by_zone',
    comment='KPIs por zona de pickup — equivalente a gold_sales_kpis por VKORG'
)
def gold_kpis_by_zone():
    return (
        dlt.read('silver_trips')
            .groupBy('pickup_zip', 'year', 'month')
            .agg(
                count('*').alias('num_viajes'),
                _round(_sum('fare_amount'), 2).alias('revenue_total'),
                _round(avg('fare_amount'), 2).alias('ticket_promedio'),
                _round(avg('trip_distance'), 2).alias('distancia_promedio')
            )
    )


## Exploración — después de correr el pipeline

Una vez que el pipeline haya corrido exitosamente,
ejecuta estas queries en SQL Editor o en un notebook separado.

```sql
-- Ver el DAG de linaje en Unity Catalog
USE CATALOG laboratory_sap_dev;
USE SCHEMA sap_course;

-- Gold KPIs por zona (equivalente a gold_sales_kpis)
SELECT *
FROM gold_kpis_by_zone
ORDER BY revenue_total DESC
LIMIT 20;

-- Calidad de datos — cuántas filas pasaron/fallaron
SELECT * FROM event_log(<pipeline_id>)
WHERE event_type = 'flow_progress'
ORDER BY timestamp DESC;
```
